In [ ]:
from pathlib import Path

import polars as pl
from loguru import logger

from scrapetube.utils.config import (
    QUERY_STRINGS,
    VIDEO_BASE_URL,
    KEYWORD_VIDEOS_META_PARQUET,
    CHANNEL_VIDEOS_META_PARQUET,
    get_timestamp_string
)
from scrapetube.cc.extractor import collect_video_metadata_concurrent
from scrapetube.utils.logger import setup_logging

pl.Config.set_fmt_str_lengths(100)

today_string = get_timestamp_string()
setup_logging(write_to_file=False)
data_dir = Path("../data")

In [ ]:
def load_latest_data(prefix:str, data_dir:Path):
    """Load the most recently modified parquet file with the given prefix from data directory."""
    latest_file = max(data_dir.glob(f"{prefix}*.parquet"), key=lambda f: f.stat().st_mtime)
    logger.info(f"Loaded {latest_file}")
    df = pl.read_parquet(latest_file)
    return df

In [ ]:
QUERY_STRINGS[:5]

In [ ]:
KEYWORD_VIDEOS_META_PARQUET

In [ ]:
params = {
    "queries": QUERY_STRINGS[:3],
    "limit": 5,
    "sleep": (3, 20),
    "sp_filter": "relevance",
    "results_type": "video",
    "proxies": None,
    "video_base_url": VIDEO_BASE_URL,
    "file_path_parquet": KEYWORD_VIDEOS_META_PARQUET,
}
results = collect_video_metadata_concurrent(**params)

In [ ]:
video_meta_df = load_latest_data("meta_data", data_dir)
unique_channel = [f'"{c}"' for c in video_meta_df["channel"].unique()]
unique_channel

In [ ]:
params = {
    "queries": unique_channel,
    "limit": 3,
    "sleep": (3, 20),
    "sp_filter": "relevance",
    "results_type": "video",
    "proxies": None,
    "video_base_url": VIDEO_BASE_URL,
    "file_path_parquet": CHANNEL_VIDEOS_META_PARQUET,
}
results = collect_video_metadata_concurrent(**params)